# Long-Running& Async Agents

**Module 11 · Lesson 11.9**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Durable State: Checkpoint and Resume
- Step Memoization: Never Re-Run Completed Work
- The Async Job Pattern: Submit, Poll, Retrieve
- Delivery Semantics: At-Least-Once + Idempotency
- Event-Driven Triggers & Backpressure
- Reliability: Circuit Breakers, Backoff & Deadlines

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

## Crash at invoice 7,431 of 10,000

> 💡 **Info**
>
> Your agent is eight hours into processing 10,000 invoices. At document 7,431 the worker crashes. Two questions decide whether this is a shrug or a disaster. First: does it **lose everything and start over** - re-reading 7,430 documents and re-spending every token - or pick up at 7,431 as if nothing happened? Second: when the retry re-delivers a task that had already **emailed a customer**, does it email them a *second* time? Long-running agents live or die on these two questions - crash recovery and duplicate side effects. This lesson builds the machinery that answers both: checkpointing and memoization so a crash costs nothing, and idempotency so a retry never double-acts. The good news: this is all engineering logic, so you can run almost every piece of it with no LLM key.

### 🎯 What you will be able to do after this lesson

- **Build** a durable agent that checkpoints every step and resumes from the last one, with step memoization so completed work never re-runs.

- **Compare** delivery semantics (at-least-once + idempotency vs the impossible exactly-once) and the stack (durable execution vs task queue vs serverless job).

- **Operate** the async job pattern (submit → poll → retrieve) and event-driven triggers with backpressure.

- **Evaluate** the reliability spine - circuit breakers, retry/backoff, dead-letter queues, deadlines - and pick the right runtime for a task.

> 📦 **Info**
>
> ✅ Before you startThis builds on **11.1** (the agent loop and its step cap), **11.3** (LangGraph checkpointing - saving graph state so a thread can resume), and **11.8** (cost and stopping, which become backpressure and deadlines here). This lesson is the **infrastructure** for long agents; human-in-the-loop approval is Lesson 11.10 and agent evaluation is Lesson 11.11.

## 60-Second Warm-Up

Flip each card and answer before revealing.

> ⏳ **Analogy**
>
> A short agent is **microwave cooking**: start, wait two minutes, done. A long-running agent is **slow-cooking a stew for six hours**: you check on it occasionally, the power might flicker, and you need to resume from exactly where it stopped - not start the stew over. It needs a **recipe bookmark** (a checkpoint), a **backup generator** (crash recovery), and someone to **stir occasionally** (monitoring). **Where it breaks down:** a stew resumes on its own; an agent only resumes if you built the checkpoint in first - durability is a design decision, not a default.

> 📦 **Info**
>
> 🚫 Two misconceptions this lesson kills
> - **“A long agent is just a slow function.”** A slow function that crashes loses everything and starts over. A long agent must checkpoint and resume, or every crash re-spends all the completed work.
> - **“Set exactly-once and forget it.”** Exactly-once is impossible across a boundary. You only get effectively-once by making the *side effects* idempotent - never the LLM output.

> 💡 **Info**
>
> ⚠️ Anti-patternRetrying a whole job on crash with no memoization, and firing blind retries into a down provider with no circuit breaker or deadline. The first re-runs every completed step and re-spends every token; the second turns a brief outage into a runaway bill. Memoize completed steps, break the circuit on repeated failure, and cap every loop with a deadline.

---

## 🎯 Concept 1: Durable State: Checkpoint and Resume

### Durable State: Checkpoint and Resume

Save progress after every step so a crash costs nothing. A fresh instance reloads the checkpoint and picks up where the last one stopped.

Start with the foundational move: a long agent must **checkpoint its state after every step**, so that if the worker crashes, a fresh instance can **reload the last checkpoint and resume** instead of starting over. The checkpoint holds the current step, the results so far, and the status; in production it lives on disk, in a GCS bucket, or in Postgres, but the shape is the same. The block below runs a durable agent that processes five steps, “crashes,” and then a brand-new instance reloads the same job and finishes the rest - with zero lost work - all deterministic and keyless.

> 🔖 **Analogy**
>
> It is a **recipe with a bookmark**. A cook working an eight-hour braise does not memorize where they are - they tick each step on the card as they finish it. If they are called away and someone else takes over, that person reads the card and continues from the last ticked step. The card *is* the checkpoint; without it, the new cook would have to start the braise over.

An agent crashes after checkpointing step 5 of 10. A fresh instance loads the same job_id. Where does it start?

**📝 `01_durable_agent.py`** - *runnable*

In [ ]:
# DURABLE STATE - checkpoint after every step, resume from the last one on crash.
# A long agent that crashes must NOT start over. We keep the checkpoint in a dict
# (a real one is on disk / GCS / Postgres) so the whole survive-a-crash shape runs with no key.

STORE = {}                                       # job_id -> saved state (stands in for durable storage)

class DurableAgent:
    def __init__(self, job_id):
        self.job_id = job_id
        self.state = STORE.get(job_id, {"step": 0, "results": [], "status": "pending"})

    def _checkpoint(self):
        STORE[self.job_id] = dict(self.state)     # persist after EVERY step

    def run(self, tasks, crash_at=None):
        start = self.state["step"]                # resume exactly where we left off
        print(f"  job {self.job_id}: starting at step {start}/{len(tasks)}")
        for i in range(start, len(tasks)):
            if crash_at is not None and i == crash_at:
                print(f"  ** CRASH at step {i} (state already saved through {self.state['step']}) **")
                return "crashed"
            self.state["results"].append(f"done:{tasks[i]}")
            self.state["step"] = i + 1
            self.state["status"] = "running"
            self._checkpoint()
        self.state["status"] = "completed"
        self._checkpoint()
        return "completed"

tasks = [f"doc{i}" for i in range(10)]
a = DurableAgent("job-42")
a.run(tasks, crash_at=5)                          # process 5, then crash
print(f"  after crash: {STORE['job-42']['step']} steps saved, status={STORE['job-42']['status']}")

b = DurableAgent("job-42")                         # a FRESH instance reloads the checkpoint
outcome = b.run(tasks)                              # resumes at step 5, finishes the rest
print(f"  final: {b.state['step']}/{len(tasks)} steps, status={b.state['status']}, outcome={outcome}")
print("Zero lost work: the crash cost nothing because state was checkpointed every step.")
# Output:
#   job job-42: starting at step 0/10
#   ** CRASH at step 5 (state already saved through 5) **
#   after crash: 5 steps saved, status=running
#   job job-42: starting at step 5/10
#   final: 10/10 steps, status=completed, outcome=completed
# Zero lost work: the crash cost nothing because state was checkpointed every step.

- `STORE` stands in for durable storage; `_checkpoint()` saves state after EVERY step.
- The first instance processes 5 steps, then “crashes” - but state through step 5 is already saved.
- A *fresh* `DurableAgent` with the same `job_id` reloads the checkpoint and starts at step 5.
- It finishes steps 5-9: zero lost work, because the crash landed after a checkpoint, not before.

#### 💡 What just happened

⚡ What just happened?A durable agent checkpoints after every step and resumes from the last one, so a crash costs nothing. Tradeoff: checkpointing adds a small write per step, but it turns a fragile long script into a restartable service. Every later pattern in this lesson builds on this move.

- Run steps, trigger a crash mid-run, and watch a fresh instance resume
- Progress resumes from the last checkpoint instead of restarting

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: Step Memoization: Never Re-Run Completed Work

### Step Memoization: Never Re-Run Completed Work

On a crash-retry, completed steps return their cached results. For LLM agents, that is the difference between re-spending every token and spending none.

Durable state stops you losing progress; **step memoization** stops you *paying for it twice*. When a workflow retries after a crash, a naive system re-runs every step from the start - and for an LLM agent, re-running a completed step means **re-spending its tokens**. Memoization records each completed step’s result, so on retry that step returns the cached value instantly and does *no* work. This is the core mechanism behind durable-execution frameworks: **Temporal** replays the workflow and completed activities return their recorded results; **Inngest** memoizes each `step.run()`. The block runs a five-step chain that crashes at step 3, then retries - and watches steps 1-2 return cached while only 3-5 actually run.

> ✅ **Analogy**
>
> It is a **checklist you do not redo**. If you are halfway through a moving checklist and get interrupted, you do not re-pack the boxes you already taped shut - you glance at the ticked items and carry on from the first unticked one. Memoization is that tick mark: a completed step is *done*, and no crash makes you redo it.

A 5-step agent crashes at step 3. With memoization, how many steps actually re-run on retry?

**📝 `02_memoize.py`** - *runnable*

In [ ]:
# STEP MEMOIZATION - completed steps never re-run on retry. This is why durable-execution
# frameworks (Temporal replay, Inngest steps) exist: after a crash, re-running a done step
# would re-spend its LLM tokens. Memoize the result and a retry returns it for free.

CACHE = {}                                         # step_id -> result (the memo / journal)
work_units = {"n": 0}                              # counts REAL work done (stands in for tokens)

def step(step_id, fn):
    if step_id in CACHE:                           # already completed -> return cached, do NO work
        print(f"  {step_id}: cached (0 work)")
        return CACHE[step_id]
    result = fn()                                  # first time -> actually run it
    work_units["n"] += 1
    CACHE[step_id] = result
    print(f"  {step_id}: ran (1 work unit)")
    return result

def run_chain(crash_at=None):
    for i in range(1, 6):
        if crash_at is not None and i == crash_at:
            print(f"  ** CRASH at step{i} **")
            raise RuntimeError("boom")
        step(f"step{i}", lambda i=i: f"result{i}")

print("First attempt (crashes at step 3):")
try:
    run_chain(crash_at=3)
except RuntimeError:
    pass
print(f"  work done so far: {work_units['n']} units\n")

print("Retry (no crash) - completed steps are memoized:")
run_chain()
print(f"\n  total work units: {work_units['n']} (5 steps, but only ran each ONCE)")
print("Steps 1-2 returned cached on retry: memoization saved re-spending their tokens.")
# Output:
# First attempt (crashes at step 3):
#   step1: ran (1 work unit)
#   step2: ran (1 work unit)
#   ** CRASH at step3 **
#   work done so far: 2 units
#
# Retry (no crash) - completed steps are memoized:
#   step1: cached (0 work)
#   step2: cached (0 work)
#   step3: ran (1 work unit)
#   step4: ran (1 work unit)
#   step5: ran (1 work unit)
#
#   total work units: 5 (5 steps, but only ran each ONCE)
# Steps 1-2 returned cached on retry: memoization saved re-spending their tokens.

- `CACHE` is the memo (a journal); `work_units` counts REAL work, standing in for tokens.
- The first attempt runs steps 1-2 (2 work units), then crashes at step 3.
- On retry, steps 1-2 print “cached (0 work)” - their results are returned for free.
- Total work across both attempts is 5 units, not 7: memoization saved re-running the two completed steps.

#### 💡 What just happened

⚡ What just happened?Step memoization returns completed steps from cache on retry, so their tokens are never re-spent. Tradeoff: you store each step’s result (and must keep it deterministic-to-replay), but you stop paying for a crash twice. This is WHY durable execution exists for LLM workloads, not just task queues.

- A 5-step chain crashes; on retry completed steps return cached (0 tokens)
- A token meter shows what memoization saved

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: The Async Job Pattern: Submit, Poll, Retrieve

### The Async Job Pattern: Submit, Poll, Retrieve

A long job must not block the request. Submit and get a job_id, run the work in the background, and poll for status until it is done.

A request that waits eight hours for an agent to finish is a broken request - it times out, holds a connection, and tells the user nothing. The fix is the **async job pattern**: the client **submits** the work and immediately gets back a `job_id`; a background worker **runs** the job and saves progress; the client **polls** a status endpoint until the job reports `completed`, then **retrieves** the result. The job moves through a small state machine - `queued → running → completed` (or `failed`) - with a progress counter. The block models exactly that: a submit that returns instantly, a worker that ticks through the work, and a poller that reads status until done.

> 🎭️ **Analogy**
>
> It is a **coat-check ticket**. You do not stand at the counter holding your coat while they find a hanger - you hand it over, get a numbered ticket, and walk away. Later you show the ticket to check the status (“still hanging”) or collect the coat. The ticket is the `job_id`; submit-and-poll is exactly a coat check for long work.

Why does the submit endpoint return a job_id immediately instead of the finished result?

**📝 `03_job_fsm.py`** - *runnable*

In [ ]:
# THE ASYNC JOB PATTERN - submit -> poll -> retrieve. A long job must not block the request:
# the client submits and gets a job_id, the work runs in the background, and the client POLLS
# for status until it is done. We model the job as a state machine you can step through.

JOBS = {}                                          # job_id -> job record (stands in for a DB)

def submit(job_id, items):                         # POST /jobs -> returns immediately
    JOBS[job_id] = {"status": "queued", "progress": 0, "total": len(items), "items": items, "results": []}
    return {"job_id": job_id, "status": "queued"}

def tick(job_id):                                  # the background worker does ONE unit of work
    j = JOBS[job_id]
    if j["progress"] >= j["total"]:
        j["status"] = "completed"; return
    j["status"] = "running"
    j["results"].append(f"done:{j['items'][j['progress']]}")
    j["progress"] += 1
    if j["progress"] == j["total"]:
        j["status"] = "completed"

def poll(job_id):                                  # GET /jobs/{id} -> current status + progress
    j = JOBS[job_id]
    return {"status": j["status"], "progress": j["progress"], "total": j["total"]}

print("Client submits, then polls until done:")
print("  submit ->", submit("job-7", ["a", "b", "c", "d", "e"]))
polls = 0
while poll("job-7")["status"] != "completed":
    tick("job-7")                                  # (a real worker runs this off the request thread)
    s = poll("job-7")
    polls += 1
    print(f"  poll {polls}: status={s['status']} progress={s['progress']}/{s['total']}")
print(f"Done after {polls} polls; the submit call returned instantly and never blocked.")
# Output:
# Client submits, then polls until done:
#   submit -> {'job_id': 'job-7', 'status': 'queued'}
#   poll 1: status=running progress=1/5
#   poll 2: status=running progress=2/5
#   poll 3: status=running progress=3/5
#   poll 4: status=running progress=4/5
#   poll 5: status=completed progress=5/5
# Done after 5 polls; the submit call returned instantly and never blocked.

- `submit` creates the job as `queued` and returns a `job_id` right away - it never blocks.
- `tick` is the background worker doing one unit of work and advancing progress.
- `poll` is the `GET /jobs/{{id}}` endpoint the client hits to read status and progress.
- The loop polls until `completed` - in production the worker runs off the request thread (Cloud Tasks, Celery, a durable workflow).

#### 💡 What just happened

⚡ What just happened?The async job pattern decouples a long job from the request: submit returns a job_id, a background worker runs and saves progress, the client polls until done. Tradeoff: the client must poll (or subscribe via SSE) instead of getting an instant answer, but the request never blocks or times out. Stream progress with SSE for a live bar.

- A job moves queued -> running -> completed while the client polls
- A progress bar fills; submit and poll are decoupled

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: Delivery Semantics: At-Least-Once + Idempotency

### Delivery Semantics: At-Least-Once + Idempotency

Exactly-once is impossible across a boundary. Make the SIDE EFFECTS idempotent with a key, and a re-delivered task acts only once.

Here is the reliability idea that trips up the most teams. Distributed queues deliver **at-least-once**: if a worker finishes a task but crashes before acknowledging it, the queue re-delivers it - so your task *will* sometimes run twice. You cannot wish this away with “exactly-once”: across a network boundary it is *provably impossible* - the acknowledgement that confirms delivery can itself be lost, so the sender can never be *certain* the message arrived and not just the ack (the **Two Generals problem**). What every serious system does instead (Stripe, Uber, Netflix) is **at-least-once delivery + idempotent consumers = effectively-once**. The trick is to make the **side effect** idempotent - attach an **idempotency key** so a repeated delivery is a no-op - and to do this on the *actions* (charge a card, send an email, write a row), never on the LLM output, which is non-deterministic by design. The block delivers the same “charge” twice and shows the naive version double-charging while the keyed version dedupes.

> 🏦 **Analogy**
>
> It is a **bank transfer reference number**. If your banking app hiccups and you tap “Send” twice, you are not charged twice - the bank sees the same reference on both requests and processes it once. The reference number is the idempotency key: the message may arrive twice, but the money moves once.

A queue re-delivers a “send invoice” task (at-least-once). How do you make sure the customer is billed only once?

**📝 `04_idempotency.py`** - *runnable*

In [ ]:
# DELIVERY SEMANTICS - exactly-once is impossible across a boundary (Two Generals). The real
# pattern is AT-LEAST-ONCE delivery + IDEMPOTENT side effects = effectively-once. Make the ACTION
# idempotent with a key, so a re-delivered message runs the side effect only ONCE.

charges = []                                       # the real-world side effect: money moved

def charge_naive(customer, amount):                # BUG: no idempotency - runs every delivery
    charges.append((customer, amount))
    return "charged"

seen_keys = set()
def charge_idempotent(key, customer, amount):      # FIX: an idempotency key dedupes retries
    if key in seen_keys:
        return "duplicate ignored"
    seen_keys.add(key)
    charges.append((customer, amount))
    return "charged"

# The queue delivers the SAME task twice (at-least-once is normal: a retry after a lost ack).
print("At-least-once delivery re-sends the same task. Watch the side effect:")
charges.clear()
print("  naive     delivery 1 ->", charge_naive("alice", 500))
print("  naive     delivery 2 ->", charge_naive("alice", 500))
print(f"  -> charges recorded: {len(charges)}  (BUG: alice paid twice)\n")

charges.clear(); seen_keys.clear()
key = "order-123"                                  # same key for both deliveries of the same task
print("  idempotent delivery 1 ->", charge_idempotent(key, "alice", 500))
print("  idempotent delivery 2 ->", charge_idempotent(key, "alice", 500))
print(f"  -> charges recorded: {len(charges)}  (FIXED: alice paid once)")
print("Make side effects idempotent (keys, UPSERT) - never the LLM output, which is non-deterministic.")
# Output:
# At-least-once delivery re-sends the same task. Watch the side effect:
#   naive     delivery 1 -> charged
#   naive     delivery 2 -> charged
#   -> charges recorded: 2  (BUG: alice paid twice)
#
#   idempotent delivery 1 -> charged
#   idempotent delivery 2 -> duplicate ignored
#   -> charges recorded: 1  (FIXED: alice paid once)
# Make side effects idempotent (keys, UPSERT) - never the LLM output, which is non-deterministic.

- The queue delivers the SAME task twice - normal at-least-once behavior after a lost ack.
- `charge_naive` has no key, so both deliveries run the side effect: two charges (the bug).
- `charge_idempotent` records the key; the second delivery sees it and returns “duplicate ignored.”
- Idempotency lives on the ACTION (a key, an UPSERT), not the LLM reasoning - that is the load-bearing distinction.

#### 💡 What just happened

⚡ What just happened?Exactly-once is impossible across a boundary; at-least-once delivery + idempotent side effects = effectively-once. Tradeoff: you must design keys and dedup windows, but you get correctness under retries. Make the actions idempotent, never the model output.

- The same message is delivered twice
- Without a key -> two side effects; with an idempotency key -> deduped to one

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: Event-Driven Triggers & Backpressure

### Event-Driven Triggers & Backpressure

Long agents run on events - a file upload, a schedule, a webhook - not on a request. And a concurrency cap keeps a flood of events from bankrupting you.

Most long agents are not called by a user waiting at a screen - they are **triggered by an event**: a file lands in a bucket (a Pub/Sub push), a schedule fires (cron), or a webhook arrives (a GitHub PR, a Slack message). An **event router** dispatches each event to the right handler. But events can arrive in *bursts*, and every handler that fires is real compute and real tokens - so you need **backpressure**: a concurrency cap that runs a fixed number at once and **holds the overflow in a queue** rather than launching a thousand agents simultaneously. The block routes six events through three handlers with a concurrency cap of two, and shows four events waiting in the queue instead of all firing at once.

> 🔔 **Analogy**
>
> It is a **smart doorbell with a bouncer**. The doorbell routes each visitor to the right person (a delivery to the kitchen, a guest to the host). But if twenty people ring at once, the bouncer only lets a couple in at a time and asks the rest to wait in line - otherwise the house is overwhelmed. Routing is the trigger; the bouncer is backpressure.

A concurrency cap of 2 receives 6 events at once. How many run immediately, and what happens to the rest?

**📝 `05_event_router.py`** - *runnable*

In [ ]:
# EVENT-DRIVEN TRIGGERS + BACKPRESSURE - long agents usually run on an EVENT (file upload,
# cron, webhook), not a request. A router dispatches each event to the right handler; a
# concurrency CAP (backpressure) holds the overflow in a queue so a flood cannot bankrupt you.

def on_upload(e):  return f"process {e['file']}"
def on_cron(e):    return f"daily report: {e['topic']}"
def on_webhook(e): return f"respond to PR #{e['pr']}"

ROUTES = {"upload": on_upload, "cron": on_cron, "webhook": on_webhook}

events = [
    {"type": "upload",  "file": "a.pdf"},
    {"type": "upload",  "file": "b.pdf"},
    {"type": "cron",    "topic": "sales"},
    {"type": "webhook", "pr": 12},
    {"type": "upload",  "file": "c.pdf"},
    {"type": "webhook", "pr": 13},
]

MAX_CONCURRENT = 2                                  # the backpressure cap
running, queue, done = [], [], []                   # in-flight, backpressure queue, finished

def admit(e):                                       # a new event arrives
    if len(running) < MAX_CONCURRENT:
        running.append(e); return "running"
    queue.append(e); return "QUEUED (backpressure)"  # cap reached -> hold it, do not fire

def drain():                                        # a slot frees up: finish one, pull the next
    e = running.pop(0)
    done.append(ROUTES[e["type"]](e))
    if queue:
        running.append(queue.pop(0))

print(f"Admit {len(events)} events with a concurrency cap of {MAX_CONCURRENT}:")
for e in events:
    label = e.get("file") or e.get("topic") or e.get("pr")
    print(f"  {e['type']:8} {str(label):7} -> {admit(e):22} (running {len(running)}, queued {len(queue)})")
peak_queued = len(queue)
while running:                                      # drain everything to completion
    drain()
print(f"\nPeak queue depth: {peak_queued} - the cap held {peak_queued} events back instead of firing all {len(events)} at once.")
print(f"All {len(done)} handled. Backpressure is reliability AND cost control - every unthrottled LLM call is real money.")
# Output:
# Admit 6 events with a concurrency cap of 2:
#   upload   a.pdf   -> running                (running 1, queued 0)
#   upload   b.pdf   -> running                (running 2, queued 0)
#   cron     sales   -> QUEUED (backpressure)  (running 2, queued 1)
#   webhook  12      -> QUEUED (backpressure)  (running 2, queued 2)
#   upload   c.pdf   -> QUEUED (backpressure)  (running 2, queued 3)
#   webhook  13      -> QUEUED (backpressure)  (running 2, queued 4)
#
# Peak queue depth: 4 - the cap held 4 events back instead of firing all 6 at once.
# All 6 handled. Backpressure is reliability AND cost control - every unthrottled LLM call is real money.

- `ROUTES` maps each event type (upload / cron / webhook) to its handler - the trigger dispatch.
- `admit` runs an event only if fewer than `MAX_CONCURRENT` are in flight; otherwise it QUEUES it.
- With a cap of 2, four events queue (peak depth 4) instead of all six firing at once.
- `drain` finishes one and pulls the next - backpressure smooths the burst into a steady stream.

#### 💡 What just happened

⚡ What just happened?Event-driven triggers run agents on uploads, schedules, and webhooks; backpressure caps concurrency and queues the overflow. Tradeoff: queued events wait a little longer, but you avoid a burst launching a thousand agents and a thousand-dollar bill. Backpressure is reliability AND cost control.

- Events stream in and dispatch to handlers
- A concurrency cap holds the overflow in a queue instead of firing all at once

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: Reliability: Circuit Breakers, Backoff & Deadlines

### Reliability: Circuit Breakers, Backoff & Deadlines

Blind retries into a down provider make an outage worse. A circuit breaker fails fast, backoff spaces retries, a DLQ catches the rest, and a deadline stops a stuck agent.

LLM providers rate-limit, time out, and go down. The naive response - retry immediately, forever - is the *worst* thing you can do: it hammers a struggling provider and runs up the bill. Production reliability is defense in depth. A **circuit breaker** per provider trips **open** after N failures and **fast-fails** instead of calling a dead API; after a cooldown it goes **half-open** to probe, and a success **closes** it. **Retry with exponential backoff and jitter** spaces the attempts. A **dead-letter queue** catches tasks that exhaust their retries (alert when it is non-empty). And a **deadline** - a max-iteration cap, a token budget, a heartbeat timeout - stops a stuck agent from running forever. The block runs a circuit breaker against a provider that is down, then recovers.

> ⚡ **Analogy**
>
> It is an **electrical fuse**. When a circuit draws too much current, the fuse *trips* - it cuts the power fast to protect the house, rather than letting the wiring overheat and start a fire. It stays open until things cool down, then you reset it. A circuit breaker does the same for a failing API: trip fast, protect the system, probe for recovery, reset.

A provider is down and 5 calls in a row fail. What should a circuit breaker (fail_max=3) do on the 4th call?

**📝 `06_circuit_breaker.py`** - *runnable*

In [ ]:
# RELIABILITY: a CIRCUIT BREAKER protects a flaky LLM provider. After N failures it trips
# OPEN and fast-fails (no more hammering a down API); after a cooldown it goes HALF-OPEN to
# probe; a success CLOSES it again. Blind retries make an outage worse - the breaker stops that.

class CircuitBreaker:
    def __init__(self, fail_max=3, cooldown=2):
        self.fail_max, self.cooldown = fail_max, cooldown
        self.fails, self.state, self.since_open = 0, "closed", 0

    def call(self, fn, tick):
        if self.state == "open":
            if tick - self.since_open >= self.cooldown:
                self.state = "half-open"            # cooldown elapsed -> probe once
            else:
                return "FAST-FAIL (open)"           # refuse instantly, protect the provider
        try:
            result = fn()
            self.fails, self.state = 0, "closed"     # success -> reset
            return result
        except Exception:
            self.fails += 1
            if self.fails >= self.fail_max:
                self.state, self.since_open = "open", tick
            return "error"

# A provider that is DOWN for ticks 0-4, then recovers.
def provider(tick):
    if tick < 5:
        raise RuntimeError("provider down")
    return "ok"

cb = CircuitBreaker(fail_max=3, cooldown=2)
print("tick | breaker state -> result")
for t in range(9):
    r = cb.call(lambda t=t: provider(t), tick=t)
    print(f"  {t}  | {cb.state:9} -> {r}")
print("\nFailures tripped the breaker OPEN (fast-fail), the cooldown half-opened it, recovery CLOSED it.")
print("Pair with retry+backoff and a dead-letter queue for tasks that still exhaust their retries.")
# Output:
# tick | breaker state -> result
#   0  | closed    -> error
#   1  | closed    -> error
#   2  | open      -> error
#   3  | open      -> FAST-FAIL (open)
#   4  | open      -> error
#   5  | open      -> FAST-FAIL (open)
#   6  | closed    -> ok
#   7  | closed    -> ok
#   8  | closed    -> ok
#
# Failures tripped the breaker OPEN (fast-fail), the cooldown half-opened it, recovery CLOSED it.
# Pair with retry+backoff and a dead-letter queue for tasks that still exhaust their retries.

- The provider is DOWN for ticks 0-4, then recovers - a typical transient outage.
- After `fail_max=3` failures the breaker trips **open** and returns FAST-FAIL without calling the provider.
- After the `cooldown` it goes **half-open** to probe; the successful probe at tick 6 **closes** it.
- Pair the breaker with retry+backoff (space the attempts) and a dead-letter queue (catch the tasks that still fail).

#### 💡 What just happened

⚡ What just happened?Reliability is defense in depth: a circuit breaker fast-fails a down provider, backoff spaces retries, a DLQ catches exhausted tasks, and a deadline stops a stuck agent. Tradeoff: these guardrails add code and a little latency on the failure path, but they turn a provider blip into a graceful degrade instead of a runaway bill.

- Failures accumulate and the breaker trips OPEN (fast-fail)
- A cooldown half-opens it; a success closes it; a fallback chain

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: The Stack: Durable Execution vs Queues vs Serverless

### The Stack: Durable Execution vs Queues vs Serverless

Pick the runtime by the task's shape - and know that the 15-minute serverless wall is gone.

Now the judgment. Three families of runtime, and the art is matching the task to the family. **Durable-execution frameworks** (Temporal, Inngest, DBOS) are for *stateful, long, crash-recoverable* workflows - they replay/memoize to the last step and support signals for human-in-the-loop, and are now well-funded, actively maintained projects with growing agent-framework integrations. **Task queues** (Celery, Dramatiq, Taskiq) are for *high-throughput independent* units - fast and simple, but with *no mid-task checkpoint*, so a crash re-runs the whole task (keep them short). **Serverless jobs** are for scheduled/event *batches* - and the old “15-minute wall” only ever applied to plain Lambda functions: **Cloud Run Jobs run up to 7 days**, **AWS Fargate / ECS tasks** have no hard duration cap, and **Azure Durable Functions** have no hard timeout on the right plan. (Plain AWS Lambda still caps at 15 minutes per invocation; for longer AWS runs you reach for Fargate, AWS Batch, or Step Functions.) So duration is rarely the deciding factor now - *state and crash-recovery* are. The runnable block routes tasks to the right family; the illustrative block shows durable execution as workflow-as-code.

> 🚗 **Analogy**
>
> It is **choosing a vehicle for the trip**. A **freight train on rails** (durable execution) is for the long, heavy, must-arrive haul - slow to set up, but it never loses the cargo. A **fleet of scooters** (task queues) is for many small independent deliveries - fast and cheap, but each rider carries little and a dropped parcel is gone. A **rental van for the weekend** (a serverless job) is for a one-off big batch. You do not take a freight train to buy milk.

You need an 8-hour stateful workflow that resumes from the last step and pauses for human approval. Which family?

**📝 `07_stack_router.py`** - *runnable*

In [ ]:
# THE STACK - choose durable execution vs a task queue vs a serverless job by the task's shape.
# Durable execution (Temporal/Inngest/DBOS) for STATEFUL, long, crash-recoverable workflows.
# Task queues (Celery/Dramatiq) for HIGH-THROUGHPUT independent units. Serverless jobs for
# scheduled/event batches. The 15-min cap is Lambda-only: Cloud Run Jobs 7d, AWS Fargate, Azure Durable.

def choose(stateful, long_running, needs_hitl, high_throughput_independent):
    if stateful and (long_running or needs_hitl):
        return ("Durable execution (Temporal / Inngest / DBOS)", "replay/memoize resumes to the last step; signals for HITL")
    if high_throughput_independent:
        return ("Task queue (Celery / Dramatiq / Taskiq)", "many independent units; no mid-task checkpoint, so keep steps short")
    if long_running:
        return ("Serverless job (Cloud Run Jobs 7d / AWS Fargate / Azure Durable)", "run a batch to completion; add your own checkpointing")
    return ("Plain async worker (FastAPI BackgroundTasks)", "small, quick, best-effort background work")

cases = [
    ("8-hour invoice pipeline with resume + approval", dict(stateful=True,  long_running=True,  needs_hitl=True,  high_throughput_independent=False)),
    ("10,000 independent classification calls",        dict(stateful=False, long_running=False, needs_hitl=False, high_throughput_independent=True)),
    ("nightly batch report over a dataset",            dict(stateful=False, long_running=True,  needs_hitl=False, high_throughput_independent=False)),
    ("fire-and-forget email summary",                  dict(stateful=False, long_running=False, needs_hitl=False, high_throughput_independent=False)),
]
print("Route each task to the right runtime:")
for name, props in cases:
    approach, why = choose(**props)
    print(f"  {name:47} -> {approach}")
    print(f"  {'':47}    ({why})")
print("\nDurable execution for stateful long workflows; queues for throughput; serverless jobs for batches.")
# Output:
# Route each task to the right runtime:
#   8-hour invoice pipeline with resume + approval  -> Durable execution (Temporal / Inngest / DBOS)
#                                                      (replay/memoize resumes to the last step; signals for HITL)
#   10,000 independent classification calls         -> Task queue (Celery / Dramatiq / Taskiq)
#                                                      (many independent units; no mid-task checkpoint, so keep steps short)
#   nightly batch report over a dataset             -> Serverless job (Cloud Run Jobs 7d / AWS Fargate / Azure Durable)
#                                                      (run a batch to completion; add your own checkpointing)
#   fire-and-forget email summary                   -> Plain async worker (FastAPI BackgroundTasks)
#                                                      (small, quick, best-effort background work)
#
# Durable execution for stateful long workflows; queues for throughput; serverless jobs for batches.

- `choose` routes by the task’s shape: stateful+long+HITL → durable execution.
- High-throughput independent units → a task queue (no mid-task checkpoint, so keep steps short).
- A long batch → a serverless job (Cloud Run Jobs 7d / AWS Fargate / Azure Durable), with your own checkpointing.
- Small, quick, best-effort work → a plain async worker - do not bring a freight train to buy milk.

**📝 `07b_durable_execution.py`** - *illustrative*

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install python-dotenv -q  # noqa


In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("OPENAI_API_KEY", "")     # platform.openai.com
os.environ.setdefault("ANTHROPIC_API_KEY", "")    # console.anthropic.com
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


- `@activity.defn` wraps the SIDE EFFECT (the LLM call); Temporal records its result for replay.
- `@workflow.defn` is the orchestration; each activity result is journaled as it completes.
- On a crash Temporal **replays**: completed activities return recorded results (no re-run, no re-spent tokens) and it resumes at the next step.
- Inngest expresses the same idea as memoized `step.run()` blocks; DBOS as `@DBOS.step` decorators - same durability, three flavours.

#### 💡 What just happened

⚡ What just happened?The stack has three families: durable execution (stateful, long, resume + HITL), task queues (high-throughput independent), and serverless jobs (batches). Tradeoff / the whole lesson: the 15-minute wall is gone, so pick by STATE and crash-recovery, not duration. Durable execution buys memoized resume at the cost of running an orchestrator; queues are simple but forget on a crash.

- Set a task's shape: stateful? long? HITL? high-throughput?
- Watch it route to durable execution / task queue / serverless job with the time limits

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

## Putting It Together

> ✅ **Info**
>
> 🧠 The whole picture
> A long agent survives crashes by **checkpointing after every step** and resuming from the last one (Step 1), and it avoids paying twice by **memoizing** completed steps so a retry re-spends no tokens (Step 2). It decouples from the request with the **submit → poll → retrieve** job pattern (Step 3), and it stays correct under retries by making **side effects idempotent** - at-least-once + idempotency = effectively-once (Step 4). It runs on **events** with **backpressure** so a burst cannot bankrupt it (Step 5), and it degrades gracefully with a **circuit breaker, backoff, a DLQ, and deadlines** (Step 6). And you pick the **runtime** by state and crash-recovery - durable execution, a task queue, or a serverless job - now that the 15-minute wall is gone (Step 7).

| Runtime | Best for | Crash recovery | Max duration |
|---|---|---|---|
| Durable execution(Temporal / Inngest / DBOS) | stateful, long, HITL workflows | replay + memoize to last step | effectively unbounded |
| Task queue(Celery / Dramatiq / Taskiq) | high-throughput independent units | re-runs the whole task | keep tasks short |
| Serverless job(Cloud Run Jobs / AWS Fargate / Azure Durable) | scheduled / event batches | your own checkpointing | hours to days |
| Plain async worker(FastAPI BackgroundTasks) | small, quick, best-effort | none | seconds to minutes |

> 📦 **Info**
>
> ➡️ Where this goes nextYou can now build a durable, resumable, idempotent long-running agent and pick its runtime. We come back to **human-in-the-loop** approval (durable signals in long workflows) in Lesson 11.10, and to **evaluating and monitoring** long agents in Lesson 11.11. Cloud cost and pricing tradeoffs are in Module 13, data-protection and DPDP-style compliance in Module 15, and the deeper GPU / Kubernetes / observability infrastructure in Module 12 and Module 14. The primary references are the [Temporal docs](https://docs.temporal.io/), the [Cloud Run Jobs timeout docs](https://docs.cloud.google.com/run/docs/configuring/task-timeout), the [circuit breaker pattern](https://martinfowler.com/bliki/CircuitBreaker.html), and the open-source [Temporal](https://github.com/temporalio/temporal) and [Inngest](https://github.com/inngest/inngest) repositories.

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Long-Running& Async Agents**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-11_9.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-11.9.html` - regenerate this notebook whenever the source HTML is updated.*
